In [1]:
import warnings
from pathlib import Path
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import joblib
import torch

# add path to project root
import sys
PROJECT_ROOT = Path.cwd()

if "notebooks" in PROJECT_ROOT.parts:
    PROJECT_ROOT = PROJECT_ROOT.parent
    
sys.path.append(str(PROJECT_ROOT))


from src.preprocess.gaming_text_preprocessor import GamingTextPreprocessor
from src.preprocess.gaming_text_labeler import GamingTextLabeler
from src.ensemble.ensemble import Ensemble
from src.model.bert_collection import BertToxicityModelCollection
from src.ensemble.score import ClassificationMetrics

# instantiate preprocessor and labeler
gaming_labeler = GamingTextLabeler()
metrics = ClassificationMetrics()



In [2]:
CONFIG = {
    'seed': 7524,
    'data_dir': PROJECT_ROOT / 'data' / 'processed_data',
    'model_dir': PROJECT_ROOT / 'models',
    'text_col': 'message',
    'label_col': 'label'
}

np.random.seed(CONFIG['seed'])
seed = CONFIG['seed']
tc, lc = CONFIG['text_col'], CONFIG['label_col']

print('CONFIG loaded. Text column:', tc)

CONFIG loaded. Text column: message


In [3]:
# load WOT train data
d = CONFIG['data_dir']

wot_train = pd.read_parquet(d / 'wot' / 'x_train.parquet')
wot_train["data_source"] = "wot"

# load DOTA training data
dota_train = pd.read_parquet(d / 'dota' / 'x_train.parquet')
dota_train["data_source"] = "dota"

# combine datasets
train = pd.concat([wot_train, dota_train], ignore_index=True)

# clean labels and create binary label column
train = gaming_labeler.convert_binary(
    train, 
    label_column=lc, 
    output_column='label_binary'
)

In [4]:
wot_val = pd.read_parquet(d / 'wot' / 'x_validation.parquet')
wot_val["data_source"] = "wot"
dota_val = pd.read_parquet(d / 'dota' / 'x_validation.parquet')
dota_val["data_source"] = "dota"

# combine holdout datasets
val = pd.concat([wot_val, dota_val], ignore_index=True)

# clean labels and create binary label column
val = gaming_labeler.convert_binary(
    val, 
    label_column=lc, 
    output_column='label_binary'
)

# Create Objects 

In [5]:
bert_binary_model = BertToxicityModelCollection(
    model_names=["dota_binary", "jigsaw_binary", "wot_binary"]
)

Loading dota_binary from jforward/bert-toxicity/dota_binary_model...
Loading jigsaw_binary from jforward/bert-toxicity/jigsaw_binary_model...
Loading wot_binary from jforward/bert-toxicity/wot_binary_model...


In [6]:
bert_binary_ensemble = Ensemble(
    model_collections=[bert_binary_model]
)

# Simple Majority Rules

In [7]:
pred_train = bert_binary_ensemble.predict_simple_majority(train[tc])
metrics.metrics(train['label_binary'], pred_train)

Predicting with SimpleMajority...
Input has 53707 samples.
dota_binary: (53707,)
jigsaw_binary: (53707,)
wot_binary: (53707,)
Collected predictions from 3 models.
Prediction matrix shape: (53707, 3)
Aggregating predictions from 3 models...


{'CV Macro F1': 0.8427718505432904,
 'CV Weighted F1': 0.8921975776052649,
 'Accuracy': 0.8993241104511516,
 'Coverage': 1.0,
 'Precision': 0.9226975570592958,
 'FPR': 0.016467541352097534,
 'FNR': 0.3704022538738457,
 'TPR': 0.6295977461261544,
 'TNR': 0.9835324586479025}

In [8]:
pred_train = bert_binary_ensemble.predict_simple_majority(val[tc])
metrics.metrics(val['label_binary'], pred_train)

Predicting with SimpleMajority...
Input has 17056 samples.
dota_binary: (17056,)
jigsaw_binary: (17056,)
wot_binary: (17056,)
Collected predictions from 3 models.
Prediction matrix shape: (17056, 3)
Aggregating predictions from 3 models...


{'CV Macro F1': 0.793306020748871,
 'CV Weighted F1': 0.8679214998907251,
 'Accuracy': 0.8775211069418386,
 'Coverage': 1.0,
 'Precision': 0.827922077922078,
 'FPR': 0.03175792075499963,
 'FNR': 0.4493927125506073,
 'TPR': 0.5506072874493927,
 'TNR': 0.9682420792450004}

# Confidence

In [11]:
thresholds = np.array([0.5, 0.9, 0.01])

res = bert_binary_ensemble.fit_weighted_confidence_majority(
    X_val=train[tc], 
    y_val=train['label_binary'], 
    score_func=metrics.score,
    thresholds=thresholds
)

Constructing confidence tensor...
Total models in ensemble: 3
Expected confidence shape: (n_samples=53707, n_classes=2)
Starting random search for n models: 3
Trial weights: (3,)
Weighted probabilities shape: (53707, 2)
Trial weights: (3,)
Weighted probabilities shape: (53707, 2)
Trial weights: (3,)
Weighted probabilities shape: (53707, 2)
Trial weights: (3,)
Weighted probabilities shape: (53707, 2)
Trial weights: (3,)
Weighted probabilities shape: (53707, 2)
Trial weights: (3,)
Weighted probabilities shape: (53707, 2)
Trial weights: (3,)
Weighted probabilities shape: (53707, 2)
Trial weights: (3,)
Weighted probabilities shape: (53707, 2)
Trial weights: (3,)
Weighted probabilities shape: (53707, 2)
Trial weights: (3,)
Weighted probabilities shape: (53707, 2)
Trial weights: (3,)
Weighted probabilities shape: (53707, 2)
Trial weights: (3,)
Weighted probabilities shape: (53707, 2)
Trial weights: (3,)
Weighted probabilities shape: (53707, 2)
Trial weights: (3,)
Weighted probabilities shape

In [14]:
res

(array([0.53242287, 0.09282784, 0.37474929]),
 0.9,
 0.9912762723392219,
 ['dota_binary', 'jigsaw_binary', 'wot_binary'])

In [16]:
combined_pred = bert_binary_ensemble.predict_weighted_confidence_majority(
    X=train[tc], 
    weights=res[0],
    threshold=res[1]
)
metrics.metrics(train['label_binary'], combined_pred)

Constructing confidence tensor...
Total models in ensemble: 3
Expected confidence shape: (n_samples=53707, n_classes=2)
Predicted labels shape: (53707,)


{'CV Macro F1': 0.9912762723392219,
 'CV Weighted F1': 0.9952684951197824,
 'Accuracy': 0.9952834487787094,
 'Coverage': 0.8171746699685329,
 'Precision': 0.9931584948688712,
 'FPR': 0.0013057316177470689,
 'FNR': 0.02230952715027361,
 'TPR': 0.9776904728497264,
 'TNR': 0.998694268382253}

In [18]:
combined_pred = bert_binary_ensemble.predict_weighted_confidence_majority(X=val[tc],
    weights=res[0],
    threshold=res[1]
)
metrics.metrics(val['label_binary'], combined_pred)

Constructing confidence tensor...
Total models in ensemble: 3
Expected confidence shape: (n_samples=17056, n_classes=2)
Predicted labels shape: (17056,)


{'CV Macro F1': 0.9166712225717368,
 'CV Weighted F1': 0.9578171949648097,
 'Accuracy': 0.9587945440262801,
 'Coverage': 0.8210014071294559,
 'Precision': 0.9093291404612159,
 'FPR': 0.014581928523263656,
 'FNR': 0.18887330528284246,
 'TPR': 0.8111266947171576,
 'TNR': 0.9854180714767363}

In [19]:
res

(array([0.53242287, 0.09282784, 0.37474929]),
 0.9,
 0.9912762723392219,
 ['dota_binary', 'jigsaw_binary', 'wot_binary'])